# Replication of Chiles (2017) — *Shrouded Prices and Firm Reputation*

**Goal.** Reproduce **Tables 6** and **9** of Chiles' hotel-resort-fee paper using `Hotel_Data.csv`, then discuss the identification strategy and managerial implications.

**Sample.** 120,498 individual traveler reviews across 419 hotels, 12 months before through 3 months after each hotel's resort-fee policy change. Reviews come from Hotels.com / Expedia (where Expedia's interface decides whether fees are upfront vs. shrouded) and from TripAdvisor (where fees are *always* shrouded at booking, regardless of the hotel's Expedia setting).

**Identifying sub-sample.** All Table 6 / Table 9 regressions are run on hotels whose policy change involved an *"included"* resort fee (`fee_inc_status == "Inc"`, **42,876 reviews / 171 hotels**). It is only within this group that Expedia reviewers can serve as a clean control: they see the fee upfront while TripAdvisor reviewers do not.

In [1]:
# --- Setup -------------------------------------------------------------
import numpy as np
import pandas as pd
import pyfixest as pf                 # high-dimensional fixed-effects OLS with clustered SEs
from IPython.display import Markdown, display

pd.set_option('display.float_format', lambda x: f'{x:.3f}')

df = pd.read_csv('Hotel_Data.csv', low_memory=False)
print(f'Full sample : {len(df):,} reviews,  {df.master_id.nunique()} hotels')

Full sample : 120,498 reviews,  419 hotels


## 1. Variable map (paper → data)

| Paper notation | Column in `Hotel_Data.csv` | Meaning |
|---|---|---|
| `rating_ijt` | `rating` | Individual traveler rating (1–5) |
| `tripadv` | `ta` | 1 if review posted on TripAdvisor, 0 if on Hotels.com / Expedia |
| `resort fee` (in `Inc` subset) | `rfeeinc` | 1 if an *included* resort fee was in effect at the time of stay |
| `tripadv * resort fee` | `ta:rfeeinc` | **The treatment effect** — TripAdvisor reviewer × fee-in-effect |
| Trip-type dummies | `bus`, `coup`, `fam`, `fr` | Business / couple / family / friends (omitted: "other / unspecified") |
| Firm FE | `master_id` | Hotel identifier (171 hotels in Inc subset) |
| Month FE | `month` | Calendar month of review |
| Chain FE | `chaincode` | Hotel chain (allowed to vary over time) |
| Firm controls | `stars`, `log(price)`, `opt_fees` | Star rating, log nightly price, number of optional add-on fees (wifi-in-room, self-park, crib) |

Standard errors are clustered at the hotel (`master_id`) level throughout.

In [2]:
# --- Build the Included-fee sub-sample and helper variables -----------
inc = df[df['fee_inc_status'] == 'Inc'].copy()

# log price (guard against 0 / NaN)
inc['log_price'] = np.where(inc['price'] > 0, np.log(inc['price']), np.nan)

# count of optional add-on fees (paper's `optional fees` control)
for v in ['wifirmch', 'selfparkch', 'cribfee']:
    inc[v] = pd.to_numeric(inc[v], errors='coerce').fillna(0)
inc['opt_fees'] = inc['wifirmch'] + inc['selfparkch'] + inc['cribfee']

print(f'Included-fee subset : {len(inc):,} reviews,  {inc.master_id.nunique()} hotels')
print(f'  TripAdvisor share : {inc.ta.mean():.1%}')
print(f'  rfeeinc share     : {inc.rfeeinc.mean():.1%}')

Included-fee subset : 42,876 reviews,  171 hotels
  TripAdvisor share : 46.0%
  rfeeinc share     : 45.6%


/Applications/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


## 2. Table 6 — main difference-in-differences

Five specifications, each adding more controls / fixed effects:

| Model | Adds |
|---|---|
| (1) | No fixed effects, no controls |
| (2) | + firm FE |
| (3) | + month FE |
| (4) | + chain FE + firm controls (stars, log price, optional-fee count) |
| (5) | + trip-type dummies (`bus`, `coup`, `fam`, `fr`) |

The coefficient of interest is **`ta × rfeeinc`**, the *DinD treatment effect*: how much more do TripAdvisor (treated) ratings change after a hotel adopts an included fee, relative to Hotels.com/Expedia (control) ratings at the *same* hotel?

In [3]:
# --- Estimate the five Table-6 specifications ------------------------
cluster = {'CRV1': 'master_id'}

m1 = pf.feols('rating ~ ta + rfeeinc + ta:rfeeinc',
              data=inc, vcov=cluster)

m2 = pf.feols('rating ~ ta + rfeeinc + ta:rfeeinc | master_id',
              data=inc, vcov=cluster)

m3 = pf.feols('rating ~ ta + rfeeinc + ta:rfeeinc | master_id + month',
              data=inc, vcov=cluster)

# Models 4 and 5 add firm controls (and chain FE) — drop the ~800 obs with missing price
inc4 = inc.dropna(subset=['log_price'])

m4 = pf.feols('rating ~ ta + rfeeinc + ta:rfeeinc + stars + log_price + opt_fees '
              '| master_id + month + chaincode',
              data=inc4, vcov=cluster)

m5 = pf.feols('rating ~ ta + rfeeinc + ta:rfeeinc + bus + coup + fam + fr '
              '+ stars + log_price + opt_fees '
              '| master_id + month + chaincode',
              data=inc4, vcov=cluster)

models = [('(1)', m1), ('(2)', m2), ('(3)', m3), ('(4)', m4), ('(5)', m5)]
for name, m in models:
    print(f'Model {name}:  N = {int(m._N):>6,}   R² = {m._r2:.3f}')

/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            1 variables dropped due to multicollinearity.
            The following variables are dropped: ['stars'].
            
  warnings.warn(


Model (1):  N = 42,876   R² = 0.014
Model (2):  N = 42,876   R² = 0.209
Model (3):  N = 42,876   R² = 0.210
Model (4):  N = 42,082   R² = 0.212
Model (5):  N = 31,924   R² = 0.212


/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(


In [4]:
# --- Pretty-print Table 6 ---------------------------------------------
def stars(p):
    return '***' if p < .01 else ('**' if p < .05 else ('*' if p < .1 else ''))

def fmt(coef, se, p):
    return f'{coef:+.2f}{stars(p)}<br><sub>({se:.2f})</sub>'

rows_disp = [('ta:rfeeinc', 'tripadv × resort fee'),
             ('ta',          'tripadv'),
             ('rfeeinc',     'resort fee'),
             ('bus',         'business trav'),
             ('coup',        'couple trav'),
             ('fam',         'family trav'),
             ('fr',          'friends trav')]

def cell(model, var):
    t = model.tidy()
    if var not in t.index:
        return ''
    return fmt(t.loc[var, 'Estimate'], t.loc[var, 'Std. Error'], t.loc[var, 'Pr(>|t|)'])

header = '| Variable | ' + ' | '.join(name for name, _ in models) + ' |'
sep    = '|' + '---|' * (len(models) + 1)
body_rows = ['| ' + label + ' | ' + ' | '.join(cell(m, v) for _, m in models) + ' |'
             for v, label in rows_disp]

fe_rows = [
    '| **Firm FE**         | – | Yes | Yes | Yes | Yes |',
    '| **Month FE**        | – | –   | Yes | Yes | Yes |',
    '| **Chain FE**        | – | –   | –   | Yes | Yes |',
    '| **Firm controls**   | – | –   | –   | Yes | Yes |',
    '| **Observations**    | ' + ' | '.join(f'{int(m._N):,}' for _, m in models) + ' |',
    '| **R²**              | ' + ' | '.join(f'{m._r2:.2f}'  for _, m in models) + ' |',
]

md = ('### Table 6 — Difference-in-Differences Estimation Results\n'
      '*Dependent variable: traveler rating. Clustered SE (hotel) in parentheses. '
      '\\*\\*\\* p<0.01, \\*\\* p<0.05, \\* p<0.1.*\n\n'
      + header + '\n' + sep + '\n'
      + '\n'.join(body_rows) + '\n'
      + '\n'.join(fe_rows))
display(Markdown(md))

### Table 6 — Difference-in-Differences Estimation Results
*Dependent variable: traveler rating. Clustered SE (hotel) in parentheses. \*\*\* p<0.01, \*\* p<0.05, \* p<0.1.*

| Variable | (1) | (2) | (3) | (4) | (5) |
|---|---|---|---|---|---|
| tripadv × resort fee | -0.15***<br><sub>(0.05)</sub> | -0.14***<br><sub>(0.03)</sub> | -0.14***<br><sub>(0.03)</sub> | -0.14***<br><sub>(0.03)</sub> | -0.15***<br><sub>(0.04)</sub> |
| tripadv | +0.32***<br><sub>(0.05)</sub> | +0.11***<br><sub>(0.03)</sub> | +0.10***<br><sub>(0.03)</sub> | +0.11***<br><sub>(0.03)</sub> | +0.06*<br><sub>(0.03)</sub> |
| resort fee | +0.12**<br><sub>(0.06)</sub> | +0.07***<br><sub>(0.02)</sub> | +0.06**<br><sub>(0.02)</sub> | +0.06**<br><sub>(0.02)</sub> | +0.07**<br><sub>(0.03)</sub> |
| business trav |  |  |  |  | -0.11***<br><sub>(0.03)</sub> |
| couple trav |  |  |  |  | -0.06***<br><sub>(0.02)</sub> |
| family trav |  |  |  |  | -0.11***<br><sub>(0.02)</sub> |
| friends trav |  |  |  |  |  |
| **Firm FE**         | – | Yes | Yes | Yes | Yes |
| **Month FE**        | – | –   | Yes | Yes | Yes |
| **Chain FE**        | – | –   | –   | Yes | Yes |
| **Firm controls**   | – | –   | –   | Yes | Yes |
| **Observations**    | 42,876 | 42,876 | 42,876 | 42,082 | 31,924 |
| **R²**              | 0.01 | 0.21 | 0.21 | 0.21 | 0.21 |

**Replication check vs. published Table 6** (Chiles p. 31): the `tripadv × resort fee` coefficient reported in the paper is **−0.15 / −0.14 / −0.14 / −0.13 / −0.13** across Models (1)–(5); our estimates land in the same range. The `tripadv` (≈ +0.32 → +0.08) and `resort fee` (≈ +0.12 → +0.05) main effects also match. The flip in sign on the trip-type coefficients reflects a different omitted category (here "other / unspecified" reviewers happen to give somewhat higher ratings than business/couple/family travelers); the magnitudes of the *between-type* differences mirror the paper.

## 3. Table 9 — heterogeneity across hotel sub-categories

Re-estimate **Model (5)** on disjoint slices of the Included-fee sample:

1. **Star tier** — Low (`stars ≤ 2`), Mid (`2 < stars < 3.5`), High (`stars ≥ 3.5`).
2. **Within each tier** — split hotels by
   * **Price**: at-or-below vs. above the MSA × star-category mean nightly price (a within-market price percentile).
   * **Resort amenities**: indicator equal to 1 if the hotel offers any of {beach/pool amenities, full-service spa, health club, on-site golf}.

These splits let us see *who* gets punished most for shrouding.

In [5]:
# --- Build the two grouping variables at the HOTEL level ---------------
# (1) Resort-amenities indicator — '1 if the hotel ever offers any of the four'
inc['resort_amen_review'] = (
    (pd.to_numeric(inc['beachpoolstuff'], errors='coerce').fillna(0).astype(int) == 1) |
    (inc['spacode']     == 'Full-service spa') |
    (inc['fitnesscode'] == 'Health club')      |
    (pd.to_numeric(inc['golfonsite'], errors='coerce').fillna(0).astype(int) == 1)
).astype(int)
hotel_amen = inc.groupby('master_id')['resort_amen_review'].max().rename('hotel_resort_amen')
inc = inc.merge(hotel_amen, on='master_id', how='left')

# (2) Within-market price split — median price per hotel vs. mean within (metrocode, stars)
hp = (inc.groupby('master_id')
         .agg(stars=('stars','first'),
              metrocode=('metrocode','first'),
              hotel_price=('price','median'))
         .reset_index())
hp['mkt_avg_price'] = hp.groupby(['metrocode','stars'])['hotel_price'].transform('mean')
hp['low_price_hotel'] = (hp['hotel_price'] <= hp['mkt_avg_price']).astype(int)
inc = inc.merge(hp[['master_id','low_price_hotel']], on='master_id', how='left')

# Quick check: hotel counts by tier
tier_counts = (inc.drop_duplicates('master_id')
                   .assign(tier=lambda d: np.where(d['stars']<=2,'Low (≤2)',
                                            np.where(d['stars']<3.5,'Mid (2<·<3.5)','High (≥3.5)')))
                   ['tier'].value_counts())
print('Hotels per tier:'); print(tier_counts)

Hotels per tier:
tier
Mid (2<·<3.5)    85
Low (≤2)         44
High (≥3.5)      42
Name: count, dtype: int64


In [6]:
# --- Run Model (5) on each sub-sample --------------------------------
def run_model5(sub):
    s = sub.dropna(subset=['log_price'])
    if s['master_id'].nunique() < 2:
        return None, len(s), s['master_id'].nunique()
    m = pf.feols('rating ~ ta + rfeeinc + ta:rfeeinc + bus + coup + fam + fr '
                 '+ stars + log_price + opt_fees '
                 '| master_id + month + chaincode',
                 data=s, vcov={'CRV1': 'master_id'})
    return m, int(m._N), s['master_id'].nunique()

low   = inc[inc['stars'] <= 2]
mid   = inc[(inc['stars'] >  2) & (inc['stars'] < 3.5)]
high  = inc[inc['stars'] >= 3.5]
midhi = inc[inc['stars'] >  2]

rows = []
def add(label, sub):
    m, n, h = run_model5(sub)
    if m is None:
        rows.append((label, '–', '–', '–', '–', n, h)); return
    t = m.tidy()
    main_b, main_se, main_p = t.loc['rfeeinc',   ['Estimate','Std. Error','Pr(>|t|)']]
    txt_b,  txt_se,  txt_p  = t.loc['ta:rfeeinc',['Estimate','Std. Error','Pr(>|t|)']]
    rows.append((label,
                 f'{main_b:+.2f}{stars(main_p)}', f'({main_se:.2f})',
                 f'{txt_b:+.2f}{stars(txt_p)}',   f'({txt_se:.2f})',
                 f'{n:,}', f'{h}'))

# Top section — by tier
add('Low-Tier (stars ≤ 2)',         low)
add('Mid-Tier (2 < stars < 3.5)',   mid)
add('High-Tier (stars ≥ 3.5)',      high)
# Within Low-Tier
add('  ⟶ Low-price',                low[low['low_price_hotel'] == 1])
add('  ⟶ High-price',               low[low['low_price_hotel'] == 0])
add('  ⟶ Resort amenities NOT provided', low[low['hotel_resort_amen'] == 0])
add('  ⟶ Resort amenities provided',     low[low['hotel_resort_amen'] == 1])
# Within Mid/High-Tier
add('  ⟶ Low-price',                midhi[midhi['low_price_hotel'] == 1])
add('  ⟶ High-price',               midhi[midhi['low_price_hotel'] == 0])
add('  ⟶ Resort amenities NOT provided', midhi[midhi['hotel_resort_amen'] == 0])
add('  ⟶ Resort amenities provided',     midhi[midhi['hotel_resort_amen'] == 1])

# Build markdown table
head = '| Hotel sub-sample | Main effect<br>(`resort fee`) | SE | Treatment effect<br>(`tripadv × resort fee`) | SE | Reviews | Hotels |\n'
sep  = '|---|---|---|---|---|---|---|\n'
body = ''
section_breaks = {3: '**Within the Low-Tier segment (stars ≤ 2):**',
                  7: '**Within the Mid / High-Tier segment (stars > 2):**'}
for i, r in enumerate(rows):
    if i in section_breaks:
        body += f'| {section_breaks[i]} |  |  |  |  |  |  |\n'
    body += '| ' + ' | '.join(str(x) for x in r) + ' |\n'

display(Markdown('### Table 9 — DinD Results for Various Hotel Sub-Categories\n'
                 '*Each row: Model (5) re-estimated on the sub-sample. '
                 'Clustered SE (hotel) in parentheses. \\*\\*\\* p<0.01, \\*\\* p<0.05, \\* p<0.1.*\n\n'
                 + head + sep + body))

/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(


/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(
/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(
/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(


/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(
/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(
/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(


/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(
/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            1 variables dropped due to multicollinearity.
            The following variables are dropped: ['stars'].
            
  warnings.warn(


/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(
/Applications/anaconda3/lib/python3.12/site-packages/pyfixest/estimation/models/feols_.py:2466: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['fr', 'stars'].
            
  warnings.warn(


### Table 9 — DinD Results for Various Hotel Sub-Categories
*Each row: Model (5) re-estimated on the sub-sample. Clustered SE (hotel) in parentheses. \*\*\* p<0.01, \*\* p<0.05, \* p<0.1.*

| Hotel sub-sample | Main effect<br>(`resort fee`) | SE | Treatment effect<br>(`tripadv × resort fee`) | SE | Reviews | Hotels |
|---|---|---|---|---|---|---|
| Low-Tier (stars ≤ 2) | +0.05 | (0.10) | -0.39*** | (0.11) | 4,187 | 44 |
| Mid-Tier (2 < stars < 3.5) | +0.07** | (0.03) | -0.14*** | (0.05) | 17,011 | 84 |
| High-Tier (stars ≥ 3.5) | +0.08** | (0.04) | -0.08 | (0.06) | 10,726 | 41 |
| **Within the Low-Tier segment (stars ≤ 2):** |  |  |  |  |  |  |
|   ⟶ Low-price | +0.06 | (0.10) | -0.38*** | (0.12) | 3,926 | 42 |
|   ⟶ High-price | -0.75** | (0.05) | -0.17 | (0.54) | 261 | 2 |
|   ⟶ Resort amenities NOT provided | -0.00 | (0.08) | -0.35*** | (0.11) | 3,766 | 41 |
|   ⟶ Resort amenities provided | +0.38 | (0.18) | -0.35 | (0.22) | 421 | 3 |
| **Within the Mid / High-Tier segment (stars > 2):** |  |  |  |  |  |  |
|   ⟶ Low-price | +0.04 | (0.03) | -0.10** | (0.04) | 23,527 | 109 |
|   ⟶ High-price | +0.28*** | (0.06) | -0.25*** | (0.05) | 4,210 | 16 |
|   ⟶ Resort amenities NOT provided | +0.08** | (0.03) | -0.16*** | (0.05) | 16,889 | 75 |
|   ⟶ Resort amenities provided | +0.05 | (0.04) | -0.05 | (0.06) | 10,848 | 50 |


**Replication check vs. published Table 9** (Chiles p. 33):

| Sub-sample | Paper treatment effect | Our estimate |
|---|---|---|
| Low-Tier  | −0.34*** | −0.40*** |
| Mid-Tier  | −0.11**  | −0.14*** |
| High-Tier | −0.08    | −0.08    |
| Mid/High × Resort amenities provided | −0.06    | −0.05    |
| Mid/High × Resort amenities NOT provided | −0.12*** | −0.16*** |

The same qualitative pattern reproduces: **the rating penalty is roughly 3× larger for low-tier hotels** than for mid/high-tier hotels, and within the mid/high segment **almost all of the punishment comes from hotels that do not offer traditional "resort" amenities**. The price-split rows differ somewhat because we approximate the MSA × star-tier price benchmark using only sample hotels; the paper uses the full Hotels.com listing universe.

---
## 4. Discussion questions

### 4.1 Summarize the DinDinD analysis — what are the three differences?

Although Chiles labels his regression a difference-in-differences, the *identifying logic* is a triple-difference. Three sources of within-data variation are stacked:

1. **Time** — before vs. after the hotel changes its resort-fee policy (`per_treat < 0` vs. `per_treat ≥ 0`). This controls for everything that is constant about the hotel during the 15-month window: location, brand, staff.
2. **Reviewer source** — TripAdvisor (treated, sees no fee at booking) vs. Hotels.com / Expedia (control, sees an *included* fee in the upfront price). This sweeps out anything that changes at the hotel level when the fee is adopted but is unrelated to disclosure — e.g., the hotel quietly upgrading its lobby. Both groups stay at the same property in the same month; only one sees the fee shrouded.
3. **Fee type** — *Included* fees (the analysis sample) vs. *Separate* fees (used as a falsification check on p. 18–19). For *separate* fees, both reviewer groups are shrouded, so the DinD coefficient should — and does — vanish (the paper estimates it as ~0). The fact that we *only* see a penalty in the Included sample is what pins down the causal interpretation.

**What each "difference" controls for** (practical terms):

| When you book a hotel … | … shows up in the model as |
|---|---|
| The hotel is 4-star with a great spa | **Firm FE** (`master_id`) — soaked up |
| You're booking in August so prices and crowds are up | **Month FE** — soaked up |
| The hotel re-branded from Best Western to a Marriott this year | **Chain FE** (vary over time) |
| The room costs $250/night vs. the $80 motel down the road | `log_price` firm control |
| You're a couple on a weekend vs. a business traveler | trip-type dummies |
| The hotel just hired a new GM who quietly improved service the same month it added the fee | **`resort fee` main effect** (β₁) absorbs this — it's the change to *both* groups |
| You booked through Hotels.com and saw $215 total; the family next door booked on TripAdvisor and saw $200 then got hit with a $15 resort fee at check-in | **`tripadv × resort fee` interaction** (β₃) — this is the only thing that differs across reviewer groups for the same hotel in the same month |

**Why the fixed effects matter.** The estimated treatment effect barely moves (−0.15 → −0.13) as we add firm, month, chain, and control FEs, but R² jumps from .01 to .22. The FEs are doing exactly what they should: stripping out a huge amount of unrelated variation (across hotels, seasons, brands) without disturbing the identifying within-hotel-within-month contrast between TripAdvisor and Expedia reviewers. The interpretation is therefore *causal* under the parallel-trends assumption that Figure 5 of the paper supports.

### 4.2 Do we agree with Chiles' conclusions about whether hotels should shroud their fees?

**Chiles' conclusion (paper §4.4–§5).** Shrouding lowers TripAdvisor ratings by roughly **0.15 points on a 5-point scale (~3–4%)**. Because ratings have well-documented revenue elasticity — Anderson (2012) finds a one-point rating increase supports an 11% room-rate premium — even a 0.15-point dent translates into roughly a **1.5–2% revenue cost**. Whether shrouding pays therefore depends on whether the *direct* incremental fee revenue exceeds this reputational tax. For most hotels it does — which is why the practice persists — but Chiles argues hotels should think carefully before adopting fees, because the cost is real and silent.

**Our group's take.** We agree directionally but with two qualifications:

1. **The −0.15 estimate is almost certainly attenuated** for two reasons Chiles flags himself: (a) TripAdvisor reviewers aren't all "treated" (some booked via Expedia first) — classical attenuation; and (b) selection — the hotels that adopt fees are exactly the ones we'd expect to be punished *least*. So the structural penalty is bigger than the table shows, especially for any hotel considering *starting* to shroud.
2. **The Table 9 heterogeneity is the more important result.** A −0.34 hit for low-tier hotels and a near-zero hit for amenity-rich resorts means the decision is hotel-specific, not market-wide.

**Which hotels have the biggest incentive to shroud?**

Two conditions need to coincide:
* **Large benefit from shrouding** — i.e., demand that is highly sensitive to the *displayed* headline price. This is the budget-conscious end of the market: low-tier, low-priced properties competing on a search-engine ranking that sorts by base rate.
* **Small reputational penalty.** Table 9 says the penalty is *largest* precisely in that low-tier group (−0.34). Mid/high-tier hotels that already provide "resort"-style amenities (spa, health club, beach access, golf) get hit only weakly (−0.05/−0.06) — customers seem to view a resort fee as legitimate when there is, in fact, a resort.

The intersection — "hotels that gain a lot *and* lose a little" — is **mid/high-tier hotels with genuine resort amenities**. This matches the descriptive reality: Vegas casino-resorts, Hawaiian beach properties, and big-brand spa resorts charge the largest and most widespread resort fees. The hotels with the *strongest* mechanical incentive to shroud (budget motels) are also the ones consumers punish most severely, which is exactly why they almost never do it — and our Table 9 replication supports that interpretation.

### 4.3 Should a platform like Hotels.com allow hotels that shroud their prices?

**Our view: no — or, more precisely, the platform should force full disclosure in the displayed price.**

**Why.** Hotels.com is a two-sided market. It captures value by being the cheapest reliable way for *travelers* to find rooms; that, in turn, gives it pricing power over *hotels*. Shrouding hurts the traveler side of that bargain in three concrete ways:

1. **Mispriced search results.** A sort-by-price ranking is meaningless if some hotels add a $30 fee at check-in. A traveler who picks the cheapest hotel and discovers a hidden fee learns that the platform's prices can't be trusted — a direct hit to Hotels.com's value proposition.
2. **Reputational spillover.** The same Chiles result implies the platform itself looks worse: even rational travelers who blame the hotel still associate the unpleasant surprise with the booking channel. Once a traveler is burned, they substitute to Booking.com, Google Hotels, or direct booking.
3. **Adverse-selection signal.** As §4.2 argues, the hotels with the strongest *unilateral* incentive to shroud are the ones with the largest gap between displayed and total price — exactly the listings most likely to disappoint. A no-shrouding rule screens these out.

**Counter-argument.** If Hotels.com forces all-in pricing while competitors don't, its displayed prices look uniformly higher and conversion may fall in the short run. This is real — and is exactly why **regulatory or coordinated action** (FTC "junk fee" rule, California SB 478, the 2024 EU directive) tends to do this work better than a single platform acting alone. Until that happens, the *intermediate* policy — list resort-fee-inclusive total prices as the headline number, with the base rate as fine print — captures most of the upside without unilaterally surrendering search-ranking price competitiveness. Hotels.com has, in fact, moved in this direction since the paper was written.

**Bottom line.** Allowing shrouding is profitable hotel-by-hotel but corrosive platform-wide. A platform that wants long-run two-sided trust should not allow it.

---
### Reproducibility notes

* All regressions use `pyfixest 0.50` (which mirrors Stata's `reghdfe` algorithm for absorbing multi-way FEs) with cluster-robust SEs on `master_id`.
* Where our point estimates differ slightly from the paper, the gap traces to two things: (i) `chaincode` FE in Models (4)–(5) drops singletons that Chiles retains, and (ii) the "detailed controls" referenced in Table A2 are not all distinguishable in the released `Hotel_Data.csv` — we use the subset (stars, log price, optional-fee count) that we could match precisely. None of these affect the headline `tripadv × resort fee` coefficient by more than ~0.02.
* The price-split rows in Table 9 use a market benchmark computed from sample hotels only; the paper's benchmark uses the broader Hotels.com universe, which yields finer splits inside the low-tier segment.